In [10]:
## Library
import os
import glob
import sys
import numpy as np
import shutil
import time
import speclite

In [11]:
start_time = time.time()

from astropy.coordinates import SkyCoord
from astropy.time import Time
from astropy import units as u
from astropy.io import fits
from astropy.table import Table
from astropy.table import vstack
from astropy.table import hstack
import warnings
warnings.filterwarnings("ignore")

In [12]:
import matplotlib.pyplot as plt
import matplotlib as mpl

mpl.rcParams["axes.titlesize"] = 14
mpl.rcParams["axes.labelsize"] = 20
plt.rcParams['savefig.dpi'] = 500
plt.rc('font', family='serif')

In [13]:
import sys
sys.path.append(os.path.join('..', 'src'))
from helper import makeSpecColors
from paths import *
from var import *
from sdtpy import *

In [14]:
path_detection_table = os.path.join(PHOT_7DT_DETECTED_DATA, "synphot_normal_class_detection.csv")

phot_table = Table.read(path_detection_table)

In [15]:
central_wavelengths = [float(filte[1:])*1e1 for filte in MEDIUM_BANDS]

In [16]:
nn = 5000
mags = [phot_table[f"magobs_{filte}"][nn] for filte in MEDIUM_BANDS]

In [17]:
import math

def write_snid_input_spectrum(wavelengths, magnitudes, filename="input_spectrum.dat"):
    """
    Writes a two-column input spectrum file for SNID from given wavelength and magnitude lists.

    Args:
        wavelengths (list): A list of wavelength values (in Angstroms).
        magnitudes (list): A list of corresponding magnitude values.
                           None or NaN values will be skipped.
        filename (str): The name of the output file.
    """
    if len(wavelengths) != len(magnitudes):
        print("Error: Wavelengths and magnitudes lists must have the same length.")
        return

    try:
        with open(filename, 'w') as f:
            # Write a header comment as per SNID format
            f.write("# SNID Input Spectrum Data\n")
            f.write(f"# Generated from Python script on {__import__('datetime').datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
            f.write("# Wavelength (Angstroms) | Magnitude\n")
            f.write("# ----------------------------------\n")

            for i in range(len(wavelengths)):
                wavelength = wavelengths[i]
                magnitude = magnitudes[i]

                # Handle None or NaN values by skipping the data point
                if magnitude is None or (isinstance(magnitude, (float, int)) and math.isnan(magnitude)):
                    # Optionally, you can log which data points were skipped
                    # print(f"Skipping data point at wavelength {wavelength} due to None/NaN magnitude.")
                    continue

                # Write the wavelength and magnitude to the file, separated by a space
                f.write(f"{wavelength} {magnitude}\n")

        print(f"Successfully created SNID input spectrum file: {filename}")

    except IOError as e:
        print(f"Error writing to file {filename}: {e}")
    except Exception as e:
        print(f"An unexpected error occurred: {e}")

# --- Example Usage ---

# Example data (replace with your actual data)
# For demonstration, let's include some None/NaN values
# example_wavelengths = [3800, 3801, 3802, 3803, 3804, 3805, 3806, 3807, 3808, 3809, 3810]
# example_magnitudes = [17.1, 17.2, None, 17.4, 17.5, math.nan, 17.7, 17.8, 17.9, 18.0, 18.1]

# Call the function to create the SNID input file
# write_snid_input_spectrum(example_wavelengths, example_magnitudes, "my_spectrum.dat")

# You can then use this file with SNID:
# snid my_spectrum.dat param=snid.param


In [18]:
for nn in range(len(phot_table)):
    mags = [phot_table[f"magobs_{filte}"][nn] for filte in MEDIUM_BANDS]
    output_data = os.path.join(PHOT_7DT_DETECTED_DATA_FOR_SNID, f"{os.path.basename(phot_table['spec'][nn])}_snid.dat")
    write_snid_input_spectrum(central_wavelengths, mags, output_data)

Successfully created SNID input spectrum file: /home/gpaek/SED-Classifier/script/../data/Synphot/Photometry/7DT/Detection/SNID/tPS1-15p_20150114_Gr13_Free_slit1.0_1_f.asci_snid.dat
Successfully created SNID input spectrum file: /home/gpaek/SED-Classifier/script/../data/Synphot/Photometry/7DT/Detection/SNID/tASASSN-15bf_20150120_Gr13_Free_slit1.0_1_f.asci_snid.dat
Successfully created SNID input spectrum file: /home/gpaek/SED-Classifier/script/../data/Synphot/Photometry/7DT/Detection/SNID/tASASSN-15cb_20150125_Gr13_Free_slit1.0_1_f.asci_snid.dat
Successfully created SNID input spectrum file: /home/gpaek/SED-Classifier/script/../data/Synphot/Photometry/7DT/Detection/SNID/tASASSN-15cd_20150127_Gr13_Free_slit1.0_1_f.asci_snid.dat
Successfully created SNID input spectrum file: /home/gpaek/SED-Classifier/script/../data/Synphot/Photometry/7DT/Detection/SNID/tASASSN-15da_20150213_Gr13_Free_slit1.0_1_f.asci_snid.dat
Successfully created SNID input spectrum file: /home/gpaek/SED-Classifier/scrip

Successfully created SNID input spectrum file: /home/gpaek/SED-Classifier/script/../data/Synphot/Photometry/7DT/Detection/SNID/tOGLE16esg_20161021_Gr13_Free_slit1.5_1_f.asci_snid.dat
Successfully created SNID input spectrum file: /home/gpaek/SED-Classifier/script/../data/Synphot/Photometry/7DT/Detection/SNID/tOGLE16etd_20161021_Gr13_Free_slit1.5_1_f.asci_snid.dat
Successfully created SNID input spectrum file: /home/gpaek/SED-Classifier/script/../data/Synphot/Photometry/7DT/Detection/SNID/tOGLE16eun_20161101_Gr13_Free_slit1.0_1_f.asci_snid.dat
Successfully created SNID input spectrum file: /home/gpaek/SED-Classifier/script/../data/Synphot/Photometry/7DT/Detection/SNID/tOGLE16euo_20161101_Gr13_Free_slit1.0_1_f.asci_snid.dat
Successfully created SNID input spectrum file: /home/gpaek/SED-Classifier/script/../data/Synphot/Photometry/7DT/Detection/SNID/tOGLE16exz_20161106_Gr13_Free_slit1.0_1_f.asci_snid.dat
Successfully created SNID input spectrum file: /home/gpaek/SED-Classifier/script/../d

In [28]:
# snid param=snid.param {spec file}
commands = []
for nn in range(len(phot_table)):
    output_data = os.path.join(PHOT_7DT_DETECTED_DATA_FOR_SNID, f"{os.path.basename(phot_table['spec'][nn])}_snid.dat")
    logfile = f"{os.path.basename(output_data)}.log"
    snidcom = f"snid param=snid.param '{os.path.basename(output_data)}' > '{logfile}'"
    commands.append(snidcom)

In [29]:
command_text = os.path.join(PHOT_7DT_DETECTED_DATA_FOR_SNID, "commands.txt")
with open(command_text, 'w') as f:
    for com in commands:
        f.write(com+"\n")